# **Ejemplo Grafico Recall Vs Precisión y curvas ROC-AUC**

In [ ]:
# Paso 1: Importar librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import precision_recall_curve, roc_curve, auc, average_precision_score, roc_auc_score

In [ ]:
# Paso 2: Cargar datos de ejemplo (dataset binario)
# Crea un conjunto de datos sintético de clasificación usando la función make_classification() del módulo sklearn.datasets
X, y = datasets.make_classification(n_samples=1000, n_features=20, n_classes=2,
                                     n_informative=2, n_redundant=10, random_state=42)


In [ ]:
# Paso 3: Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:
# Paso 4: Normalizar los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**COMENTARIO**

En los modelos SVC (Support Vector Classifier) de scikit-learn, el clasificador por defecto no calcula probabilidades de pertenencia a cada clase.
Solo entrega una decisión categórica (por ejemplo: clase 0 o clase 1).

Sin embargo, en muchos casos (como ajustar umbrales de decisión, **curvas ROC**, o calcular métricas probabilísticas), sí necesitamos esas probabilidades.
Por eso existe el parámetro: `probability=True`

In [ ]:
# Paso 5: Entrenar un modelo SVM con probabilidad
svm = SVC(kernel='linear', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

In [ ]:
# Paso 6: Predecir probabilidades
y_scores = svm.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Paso 7: Calcular curvas
precision, recall, _ = precision_recall_curve(y_test, y_scores)
fpr, tpr, _ = roc_curve(y_test, y_scores)

In [ ]:
# Paso 8: Calcular áreas bajo la curva
roc_auc = auc(fpr, tpr)
pr_auc = average_precision_score(y_test, y_scores)

print(f"roc_auc:  {roc_auc:.2f}")
print(f"pr_auc:  {pr_auc:.2f}")

In [ ]:
# Paso 9: Visualizar curvas
plt.figure(figsize=(12, 5))

# Curva Precisión vs Recall
plt.subplot(1, 2, 1)
plt.plot(recall, precision, label=f'AP = {pr_auc:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision vs Recall')
plt.grid(True)
plt.legend()

# Curva ROC
plt.subplot(1, 2, 2)
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')  # Línea diagonal
plt.xlabel('FPR (1 - Specificity)')
plt.ylabel('TPR (Recall)')
plt.title('Curva ROC')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


**Análisis de la curva Precision-Recall (izquierda)**

Esta curva muestra cómo cambia la precisión del modelo cuando aumenta la proporción de casos positivos identificados (recall).
- Un AP de 0.84 indica que el modelo mantiene, en términos generales, una precisión elevada al recuperar los casos positivos.
- **Conclusión:** el modelo consigue un buen equilibrio entre precision y recall en una parte importante de la curva. Sin embargo, al intentar detectar prácticamente todos los positivos, la precisión disminuye considerablemente.

**Análisis de la curva ROC (derecha)**

La curva ROC representa la relación entre:
- TPR (Recall): proporción de positivos correctamente identificados.
- FPR: proporción de negativos incorrectamente clasificados como positivos.

La gráfica muestra que **la curva azul** se encuentra claramente por encima de la diagonal punteada, que representa un clasificador sin capacidad discriminativa.

**¿Qué significa AUC = 0.90?**

Significa que, al seleccionar aleatoriamente un caso positivo y uno negativo, existe aproximadamente un 90 % de probabilidad de que el modelo asigne una puntuación mayor al caso positivo.

Por tanto, el modelo tiene una excelente capacidad para diferenciar ambas clases, aunque eso no garantiza que todos los umbrales produzcan buenas decisiones.

**la idea fundamental es:**
- ROC responde: ¿qué tan bien distingue el modelo las dos clases?
- Precision-Recall responde: ¿qué tan confiables son las predicciones positivas a medida que intentamos recuperar más casos positivos?